# 31. Choosing the two players from every tracked person

`two_player/dump_tracks.py` saved the boxes of **every** person the tracker followed in every clip (lower detection threshold, larger image size, so small far players are found). Notebook 30 showed why the earlier two player pick failed:
it ranked the second player by box size, but the real opponent at the far end of the court is the smallest person in the frame, so a larger bystander won.

This notebook picks the two **players** by where they stand and how they move.

1. **Merge track fragments.** The tracker sometimes drops a person and starts a new id a few frames later at the same spot. Fragments that follow each other closely are joined.
2. **A court region per source video.** Each of the 80 videos has one fixed camera and 67 to 266 clips. Players move across the court in all of them, so a heat map of where *moving* people's feet are, over all of a video's clips, outlines the playing area. Static bystanders and people on the edge of the hall add little.
3. **Score every track in a clip:** how long it is visible, how much of that time its feet are inside the court region, and how much it moves.
4. **Pick the best track, then the best other track that is on screen at the same time and is a different person** (not a fragment, boxes that do not overlap). Person 0 is the one nearer the camera (lower feet), person 1 the other.
5. **Check by eye**: the court region on a frame of every video, and a random sample of clips with all tracks drawn and the chosen two highlighted.

Nothing is decided from labels or from the test set. Everything is geometry from the boxes. The output is `boxes_selected/` (per frame boxes for both players, NaN where absent) and `selected_tracks.csv`, which the pose step reads.

In [ ]:
# Paper figure: wrong player examples, drawn at the printed width.
# Read only: it loads saved audit files and the stored skeletons, and writes one PNG. Nothing else is touched.
import glob, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
try:
    import cv2
except ImportError:
    cv2 = None

REPO = os.environ.get("BADMINTON_REPO", "/workspace/badminton-honours-project")
DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
RAW_VIDEOS = os.environ.get("BADMINTON_RAW_VIDEOS", f"{DATA}/raw")
sys.path.insert(0, REPO)
import dataset as ds

SEL_RAW, SEL_BOX = f"{DATA}/keypoints_sel_raw", f"{DATA}/boxes_sel"
FIG_DIR = f"{REPO}/paper_figures"; os.makedirs(FIG_DIR, exist_ok=True)
N_TARGET, CONF_MIN = 30, 0.3
NEAR, FAR, OLD, ACT = "#4c72b0", "#dd8452", "#c0392b", "#2e8b57"
N_EXAMPLES = 2                                    # rows in the figure
FIG_W, ROW_H = 4.8, 1.62                          # inches: drawn at the printed width (LNCS text width), include at width=\textwidth
SLOTS = (2, 10, 18, 26)                           # frames shown, left to right in time
TITLE_SIZE, TAG_SIZE, LEGEND_SIZE = 7, 5.5, 5.5   # points, as printed
TAG = dict(boxstyle="round,pad=0.2", fc="white", alpha=0.92, lw=0.5)

# the same clips as notebook 35: wrong clips of the random audit sample, first clip of each class, in audit order
v = pd.read_csv(f"{OUTPUTS}/results/wrong_player_audit_verdicts.csv")
wrong = v[v["actor"].isin(["near", "far"]) & (v["old_is"] != "unknown") & (v["old_is"] != v["actor"])]
chosen = wrong.drop_duplicates("cls").head(N_EXAMPLES)
print(chosen[["number", "cls", "old_is", "actor", "filepath"]].to_string(index=False))

logs = [f for f in sorted(glob.glob(f"{DATA}/extraction_sel_log*.csv")) if os.path.getsize(f) > 0]
sel = pd.concat([pd.read_csv(f) for f in logs], ignore_index=True).drop_duplicates("filepath", keep="last").set_index("filepath")
manifest = ds.load_filtered_manifest(f"{DATA}/split_manifest.csv", f"{DATA}/extraction_log.csv",
                                     f"{DATA}/keypoints_interp_log.csv", 5).set_index("filepath")


def stem_of(fp):
    return os.path.splitext(os.path.basename(fp))[0]


def draw_person(ax, kp, box, colour):
    if box is None or np.isnan(box[0]):
        return
    vis = kp[:, 2] > CONF_MIN
    for a, b in ds.COCO_SKELETON:
        if vis[a] and vis[b]:
            ax.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], color=colour, lw=0.8)
    ax.scatter(kp[vis, 0], kp[vis, 1], s=2.5, color=colour, zorder=3)
    ax.add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec=colour, lw=0.6, ls="--"))


def tag(ax, b, p, text, colour):
    """Label with a white background: above the near player's box, below the far player's box."""
    if p == 0:
        ax.text(b[0], b[1] - 8, text, color=colour, fontsize=TAG_SIZE, weight="bold", va="bottom", ha="left", bbox={**TAG, "ec": colour}, zorder=10)
    else:
        ax.text(b[0], b[3] + 8, text, color=colour, fontsize=TAG_SIZE, weight="bold", va="top", ha="left", bbox={**TAG, "ec": colour}, zorder=10)


def draw_example(axes, fp, old_is, actor):
    cls, stem = manifest.loc[fp, "class_name"], stem_of(fp)
    kps = np.load(f"{SEL_RAW}/{cls}/{stem}.npy")
    boxes = np.load(f"{SEL_BOX}/{cls}/{stem}.npy")
    targets = np.linspace(sel.loc[fp, "span_first"], sel.loc[fp, "span_last"], N_TARGET).round().astype(int)
    path = f"{RAW_VIDEOS}/{fp}"
    cap = cv2.VideoCapture(path) if (cv2 is not None and os.path.exists(path)) else None
    okb = boxes[~np.isnan(boxes[:, :, 0])]
    x0, y0 = okb[:, :2].min(axis=0)
    x1, y1 = okb[:, 2:].max(axis=0)
    mx, my = 0.06 * (x1 - x0), 0.14 * (y1 - y0)
    for k, (ax, t) in enumerate(zip(axes, SLOTS)):
        if cap is not None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(targets[t]))
            ok, bgr = cap.read()
            if ok:
                ax.imshow(bgr[..., ::-1])
        draw_person(ax, kps[0, t], boxes[0, t], NEAR)
        draw_person(ax, kps[1, t], boxes[1, t], FAR)
        for name, p in (("near", 0), ("far", 1)):
            b = boxes[p, t]
            if np.isnan(b[0]):
                continue
            if actor == name:                                            # who really hits the shot: green
                ax.add_patch(Rectangle((b[0] - 3, b[1] - 3), b[2] - b[0] + 6, b[3] - b[1] + 6, fill=False, ec=ACT, lw=1.2, zorder=8))
                if k == 0:
                    tag(ax, b, p, "hits the shot", ACT)
            if old_is == name:                                           # what the old pipeline tracked: red
                ax.add_patch(Rectangle((b[0] - 3, b[1] - 3), b[2] - b[0] + 6, b[3] - b[1] + 6, fill=False, ec=OLD, lw=1.2, zorder=8))
                if k == 0:
                    tag(ax, b, p, "old pipeline", OLD)
        ax.set_xlim(x0 - mx, x1 + mx)
        ax.set_ylim(y1 + my, y0 - my)
        ax.set_aspect("equal")
        ax.axis("off")
    if cap is not None:
        cap.release()


def crop_aspect(fp):
    """width / height of the region shown for a clip (the same limits draw_example uses)."""
    cls, stem = manifest.loc[fp, "class_name"], stem_of(fp)
    boxes = np.load(f"{SEL_BOX}/{cls}/{stem}.npy")
    okb = boxes[~np.isnan(boxes[:, :, 0])]
    x0, y0 = okb[:, :2].min(axis=0)
    x1, y1 = okb[:, 2:].max(axis=0)
    return (1.12 * (x1 - x0)) / (1.28 * (y1 - y0))


# layout in inches so every frame fills its cell exactly: no empty margins, no legend
GAP_X, TITLE_H, GAP_Y = 0.04, 0.17, 0.03
cell_w = (FIG_W - (len(SLOTS) - 1) * GAP_X) / len(SLOTS)
aspects = [crop_aspect(fp) for fp in chosen["filepath"]]
row_h = [cell_w / a for a in aspects]
H = sum(h + TITLE_H + GAP_Y for h in row_h)
fig = plt.figure(figsize=(FIG_W, H))
y_top = H
for (_, r), h in zip(chosen.iterrows(), row_h):
    y_top -= TITLE_H
    row = [fig.add_axes([k * (cell_w + GAP_X) / FIG_W, (y_top - h) / H, cell_w / FIG_W, h / H]) for k in range(len(SLOTS))]
    draw_example(row, r["filepath"], r["old_is"], r["actor"])
    row[0].set_title(f"{r['cls']}: the old pipeline tracked the {r['old_is']} player, but the {r['actor']} player hits the shot",
                     fontsize=TITLE_SIZE, loc="left", weight="normal", pad=2)
    y_top -= h + GAP_Y
fig.savefig(f"{FIG_DIR}/wrong_player_examples.png", dpi=400, facecolor="white")     # exactly FIG_W wide, no cropping needed
plt.show()
print("saved", f"{FIG_DIR}/wrong_player_examples.png", f"({FIG_W} x {H:.2f} in)")

In [ ]:
TITLE_SIZE = 17
fig, axs = plt.subplots(len(chosen), len(SLOTS), figsize=(3.2 * len(SLOTS), 3.9 * len(chosen)), squeeze=False)
for row, (_, r) in zip(axs, chosen.iterrows()):
    draw_example(row, r["filepath"], r["old_is"], r["actor"])
    row[0].set_title(f"{r['cls']}: the old pipeline tracked the {r['old_is']} player, but the {r['actor']} player hits the shot",
                     fontsize=TITLE_SIZE, loc="left", weight="normal", pad=12)
fig.legend(handles=[Patch(fc="none", ec=OLD, lw=2.6, label="player tracked by the old pipeline"),
                    Patch(fc="none", ec=ACT, lw=2.6, label="player who hits the shot"),
                    Patch(fc=NEAR, label="near player skeleton"), Patch(fc=FAR, label="far player skeleton")],
           loc="lower center", ncol=4, frameon=False, fontsize=11)
fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.07, wspace=0.03, hspace=0.45)
fig.savefig(f"{FIG_DIR}/wrong_player_examples.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("saved", f"{FIG_DIR}/wrong_player_examples.png")

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from clearml import Task

try:
    import cv2
except ImportError:
    cv2 = None

DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
RAW_VIDEOS = os.environ.get("BADMINTON_RAW_VIDEOS", f"{DATA}/raw")
EXPORT = f"{OUTPUTS}/results"
os.makedirs(EXPORT, exist_ok=True)
TRACKS, BOXES_SEL = f"{DATA}/tracks_all", f"{DATA}/boxes_selected"

# parameters, written down here so the result can be reproduced
PERSIST = 0.3          # a track must be visible in this share of the clip's frames to be a candidate
MERGE, MERGE_GAP, MERGE_DIST = True, 8, 1.0   # join fragments: new track starts within 8 frames, within 1 body height of where the old one ended
GX, GY = 96, 54        # court region grid cells over the frame
TRAVEL_MIN = 0.5       # path length / body height a track needs to count as a mover when building the court region
TRAVEL_FULL = 0.8      # path length / body height at which the movement factor reaches 1
REL = 0.04             # court region = cells with at least this fraction of the busiest cell's movement
INSIDE_MIN = 0.5       # share of a track's frames with feet inside the court region to be a candidate player
MIN_COOCCUR, MAX_IOU = 0.3, 0.3    # the second player must share this much of its frames with the first, without overlapping it
NEAR, FAR = "#4c72b0", "#dd8452"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

task = Task.init(project_name="BadmintonPoseAnalysis/results", task_name="two_player_selection", task_type=Task.TaskTypes.data_processing,
                 tags=["analysis", "two_player"], reuse_last_task_id=False, auto_connect_frameworks={"matplotlib": False})
logger = task.get_logger()
task.connect({"PERSIST": PERSIST, "MERGE": MERGE, "MERGE_GAP": MERGE_GAP, "MERGE_DIST": MERGE_DIST, "GRID": [GX, GY], "TRAVEL_MIN": TRAVEL_MIN,
              "TRAVEL_FULL": TRAVEL_FULL, "REL": REL, "INSIDE_MIN": INSIDE_MIN, "MIN_COOCCUR": MIN_COOCCUR, "MAX_IOU": MAX_IOU}, name="selection")

## 1. Load every clip's tracks

In [ ]:
manifest = pd.read_csv(f"{DATA}/split_manifest.csv")
manifest["stem"] = manifest["filepath"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])
manifest["video"] = manifest["stem"].str.split("_dataset").str[0]
manifest["has_tracks"] = [os.path.exists(f"{TRACKS}/{c}/{s}.npz") for c, s in zip(manifest["class_name"], manifest["stem"])]
print(f"track files found for {manifest['has_tracks'].sum()} of {len(manifest)} clips")
if not manifest["has_tracks"].all():
    print("WARNING: the dump is not finished. Continuing with the clips that are done; the court regions will be rougher for videos with few clips done.")
manifest = manifest[manifest["has_tracks"]].reset_index(drop=True)


def load_tracks(class_name, stem):
    z = np.load(f"{TRACKS}/{class_name}/{stem}.npz")
    rows = z["rows"]
    tracks = {}
    for tid in np.unique(rows[:, 0]):
        r = rows[rows[:, 0] == tid]
        r = r[np.argsort(r[:, 1])]
        tracks[int(tid)] = {"f": r[:, 1].astype(int), "b": r[:, 2:6].astype(np.float32), "ids": [int(tid)]}
    return tracks, int(z["n_frames"]), int(z["height"]), int(z["width"])


def feet(b):
    return np.stack([(b[:, 0] + b[:, 2]) / 2, b[:, 3]], axis=1)


def merge_fragments(tracks):
    # join a track that starts shortly after another ends, close to where it ended (same person, new tracker id)
    order = sorted(tracks, key=lambda t: tracks[t]["f"][0])
    merged, n_joins = {}, 0
    for t in order:
        tr = tracks[t]
        target = None
        for m, a in merged.items():
            gap = tr["f"][0] - a["f"][-1]
            if 0 < gap <= MERGE_GAP:
                h = np.median(a["b"][:, 3] - a["b"][:, 1])
                if h > 0 and np.linalg.norm(feet(a["b"][-1:])[0] - feet(tr["b"][:1])[0]) < MERGE_DIST * h:
                    target = m
                    break
        if target is None:
            merged[t] = {"f": tr["f"].copy(), "b": tr["b"].copy(), "ids": list(tr["ids"])}
        else:
            a = merged[target]
            a["f"] = np.concatenate([a["f"], tr["f"]])
            a["b"] = np.concatenate([a["b"], tr["b"]])
            a["ids"] += tr["ids"]
            n_joins += 1
    return merged, n_joins


def features(tr, n_frames, width, height):
    b, f = tr["b"], tr["f"]
    h = float(np.median(b[:, 3] - b[:, 1]))
    c = np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2], axis=1)
    travel = float(np.linalg.norm(np.diff(c, axis=0), axis=1).sum() / h) if len(b) > 1 and h > 0 else 0.0
    return {"cov": len(f) / max(n_frames, 1), "h": h, "travel": travel, "foot_y": float(np.median(b[:, 3]))}


clips, n_joins_total = {}, 0
for row in manifest.itertuples():
    tracks, n_frames, height, width = load_tracks(row.class_name, row.stem)
    n_raw = len(tracks)
    if MERGE and tracks:
        tracks, nj = merge_fragments(tracks)
        n_joins_total += nj
    for tr in tracks.values():
        tr.update(features(tr, n_frames, width, height))
    clips[row.filepath] = {"tracks": tracks, "n_frames": n_frames, "w": width, "h": height, "n_raw": n_raw}
print(f"loaded {len(clips)} clips; {n_joins_total} fragments joined")
print("tracks per clip after merging:", pd.Series([len(c['tracks']) for c in clips.values()]).describe().round(1).to_dict())

## 2. Court region per source video

For each video: every frame's foot position of every track that is visible in at least 40% of its clip and moves (path length of at least half a body height). These points are binned on a grid, smoothed, and thresholded.
A moving bystander contributes too, but far less than the two players, who are in every clip.

In [ ]:
heat = {}
for fp, c in clips.items():
    v = manifest.loc[manifest["filepath"] == fp, "video"].iloc[0] if False else None
vid_of = dict(zip(manifest["filepath"], manifest["video"]))
size_of_video = {}
for fp, c in clips.items():
    v = vid_of[fp]
    size_of_video[v] = (c["w"], c["h"])
    H = heat.setdefault(v, np.zeros((GY, GX)))
    for tr in c["tracks"].values():
        if tr["cov"] >= 0.4 and tr["travel"] >= TRAVEL_MIN:
            p = feet(tr["b"])
            ix = np.clip((p[:, 0] / c["w"] * GX).astype(int), 0, GX - 1)
            iy = np.clip((p[:, 1] / c["h"] * GY).astype(int), 0, GY - 1)
            np.add.at(H, (iy, ix), 1.0)

masks = {}
for v, H in heat.items():
    S = ndimage.gaussian_filter(H, sigma=1.5)
    m = S >= REL * S.max() if S.max() > 0 else np.zeros_like(S, dtype=bool)
    m = ndimage.binary_fill_holes(ndimage.binary_dilation(m, iterations=2))
    masks[v] = m
share = pd.Series({v: m.mean() for v, m in masks.items()})
print(f"court region covers {share.describe().round(2).to_dict()} of the frame, over {len(masks)} videos")
np.savez_compressed(f"{DATA}/video_court_regions.npz", **{v: m for v, m in masks.items()})

In [ ]:
videos = sorted(masks)
first_clip = manifest.drop_duplicates("video").set_index("video")["filepath"]


def video_frame(v):
    path = f"{RAW_VIDEOS}/{first_clip[v]}"
    if cv2 is None or not os.path.exists(path):
        return None
    cap = cv2.VideoCapture(path); ok, bgr = cap.read(); cap.release()
    return bgr[..., ::-1] if ok else None


def show_regions(vs, cols=4):
    rows_ = int(np.ceil(len(vs) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(4.2 * cols, 2.5 * rows_))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, v in zip(np.atleast_1d(axes).ravel(), vs):
        img = video_frame(v)
        w, h = size_of_video[v]
        if img is not None:
            ax.imshow(img, extent=(0, w, h, 0))
        ax.imshow(masks[v], extent=(0, w, h, 0), alpha=0.35, cmap="autumn", interpolation="nearest")
        ax.set_title(f"{v}  ({int((manifest['video'] == v).sum())} clips)", fontsize=7)
    plt.tight_layout(); plt.show()


show_regions(videos[:20])

In [ ]:
show_regions(videos[20:40])

In [ ]:
show_regions(videos[40:60])

In [ ]:
show_regions(videos[60:80])

How to read these: the shaded area should cover the court and the space the players move in, and not the spectators, the benches or the wall. Videos with only a handful of clips (the 2023-05-02 drill session) can have a rough or patchy region; note any that look wrong.
If a region is clearly wrong, say which videos and I will give those a manual polygon.

## 3. Score the tracks and pick the two players

In [ ]:
def inside_share(tr, mask, w, h):
    p = feet(tr["b"])
    ix = np.clip((p[:, 0] / w * GX).astype(int), 0, GX - 1)
    iy = np.clip((p[:, 1] / h * GY).astype(int), 0, GY - 1)
    return float(mask[iy, ix].mean())


def iou_on_shared(a, b):
    shared, ia, ib = np.intersect1d(a["f"], b["f"], return_indices=True)
    if len(shared) == 0:
        return 0.0, 0.0
    A, B = a["b"][ia], b["b"][ib]
    ix = np.clip(np.minimum(A[:, 2], B[:, 2]) - np.maximum(A[:, 0], B[:, 0]), 0, None)
    iy = np.clip(np.minimum(A[:, 3], B[:, 3]) - np.maximum(A[:, 1], B[:, 1]), 0, None)
    inter = ix * iy
    union = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1]) + (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1]) - inter
    return len(shared) / len(b["f"]), float(np.mean(inter / np.maximum(union, 1e-6)))


def select_pair(tracks, mask, w, h):
    cand = []
    for t, tr in tracks.items():
        ins = inside_share(tr, mask, w, h)
        tr["inside"] = ins
        if tr["cov"] >= PERSIST and ins >= INSIDE_MIN:
            tr["score"] = tr["cov"] * ins * (0.3 + 0.7 * min(1.0, tr["travel"] / TRAVEL_FULL))
            cand.append(t)
    cand.sort(key=lambda t: -tracks[t]["score"])
    if not cand:
        return []
    first = cand[0]
    for t in cand[1:]:
        co, iou = iou_on_shared(tracks[first], tracks[t])
        if co >= MIN_COOCCUR and iou < MAX_IOU:
            pair = [first, t]
            break
    else:
        pair = [first]
    return sorted(pair, key=lambda t: -tracks[t]["foot_y"])      # person 0 = nearer the camera (lower feet)


# self test: a mover inside the court, a static person at the edge (a bystander), a fragmented near player
mask = np.zeros((GY, GX), dtype=bool); mask[20:50, 10:80] = True
w, h = 960, 540
def mk(f, x, y, hh=100, dx=0.0):
    f = np.asarray(f); b = np.zeros((len(f), 4), dtype=np.float32)
    for i, fi in enumerate(f):
        cx = x + dx * i
        b[i] = [cx - hh / 4, y - hh, cx + hh / 4, y]
    return {"f": f, "b": b, "ids": [0]}
near = mk(range(0, 20), 300, 480, 120, dx=6); near2 = mk(range(23, 40), 420, 480, 120, dx=6)
far_ = mk(range(0, 40), 500, 300, 60, dx=4); bystander = mk(range(0, 40), 900, 150, 80)
tr_ = {1: near, 2: near2, 3: far_, 4: bystander}
merged, nj = merge_fragments(tr_)
assert nj == 1 and len(merged) == 3, (nj, len(merged))
for t in merged.values():
    t.update(features(t, 40, w, h))
pair = select_pair(merged, mask, w, h)
assert len(pair) == 2 and 4 not in [merged[p]["ids"][0] for p in pair] and merged[pair[0]]["foot_y"] > merged[pair[1]]["foot_y"], pair
print("selection self test passed: bystander skipped, fragments joined, near first")

In [ ]:
os.makedirs(BOXES_SEL, exist_ok=True)
rows, selected = [], {}
for row in manifest.itertuples():
    c = clips[row.filepath]
    v = row.video
    pair = select_pair(c["tracks"], masks[v], c["w"], c["h"])
    boxes = np.full((2, c["n_frames"], 4), np.nan, dtype=np.float32)
    rec = {"filepath": row.filepath, "class_name": row.class_name, "split": row.split, "video": v, "n_frames": c["n_frames"],
           "n_tracks_raw": c["n_raw"], "n_tracks_merged": len(c["tracks"]), "n_persons": len(pair)}
    for p, t in enumerate(pair):
        tr = c["tracks"][t]
        ok = tr["f"] < c["n_frames"]
        boxes[p, tr["f"][ok]] = tr["b"][ok]
        rec.update({f"p{p}_id": tr["ids"][0], f"p{p}_ids_merged": len(tr["ids"]), f"p{p}_cov": tr["cov"], f"p{p}_inside": tr["inside"],
                    f"p{p}_travel": tr["travel"], f"p{p}_score": tr["score"], f"p{p}_height": tr["h"]})
    os.makedirs(f"{BOXES_SEL}/{row.class_name}", exist_ok=True)
    np.save(f"{BOXES_SEL}/{row.class_name}/{row.stem}.npy", boxes)
    rows.append(rec)
sel = pd.DataFrame(rows)
sel.to_csv(f"{DATA}/selected_tracks.csv", index=False)
print("clips by number of players selected:", sel["n_persons"].value_counts().sort_index().to_dict())
display(sel[["p0_cov", "p0_inside", "p0_travel", "p1_cov", "p1_inside", "p1_travel", "p0_height", "p1_height"]].describe().round(2))
by_video = sel.groupby("video")["n_persons"].apply(lambda s: (s < 2).mean()).sort_values(ascending=False)
print("videos with the highest share of clips with fewer than two players:")
print(by_video.head(10).round(2).to_string())
logger.report_table("selection summary", "two_player", iteration=0, table_plot=sel["n_persons"].value_counts().rename_axis("n_persons").reset_index())

## 4. Look at the choice

Grey boxes are every tracked person at the middle frame, with the tracker's id. Blue is the chosen near player, orange the chosen far player. A good choice has blue and orange on the two players.

In [ ]:
def show_selection(fp, ax, tag=""):
    c = clips[fp]
    row = manifest[manifest["filepath"] == fp].iloc[0]
    mid = c["n_frames"] // 2
    path = f"{RAW_VIDEOS}/{fp}"
    if cv2 is not None and os.path.exists(path):
        cap = cv2.VideoCapture(path); cap.set(cv2.CAP_PROP_POS_FRAMES, mid); ok, bgr = cap.read(); cap.release()
        if ok:
            ax.imshow(bgr[..., ::-1])
    for t, tr in c["tracks"].items():
        k = np.where(tr["f"] == mid)[0]
        if len(k):
            x1, y1, x2, y2 = tr["b"][k[0]]
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#999999", lw=0.8))
            ax.text(x1, y1 - 2, str(tr["ids"][0]), color="#dddddd", fontsize=6)
    boxes = np.load(f"{BOXES_SEL}/{row.class_name}/{row.stem}.npy")
    for p, col in ((0, NEAR), (1, FAR)):
        if mid < boxes.shape[1] and not np.isnan(boxes[p, mid, 0]):
            x1, y1, x2, y2 = boxes[p, mid]
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=col, lw=2.2))
    ax.set_xlim(0, c["w"]); ax.set_ylim(c["h"], 0); ax.axis("off")
    ax.set_title(f"{tag} {row.class_name.split('_', 1)[1]}  |  {row.video}", fontsize=7)


def grid(fps, cols=4, tag=""):
    rows_ = int(np.ceil(len(fps) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(4.4 * cols, 2.7 * rows_))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for i, (ax, fp) in enumerate(zip(np.atleast_1d(axes).ravel(), fps)):
        show_selection(fp, ax, tag=f"{tag}#{i}")
    plt.tight_layout(); plt.show()


target = "2022-08-30_18-18-31_dataset_set1_046_004321_004347"
hit = [fp for fp in clips if target in fp]
if hit:
    print("the clip where the earlier pick chose a bystander:")
    grid(hit, cols=1)

In [ ]:
rng = np.random.RandomState(0)
sample_match = rng.choice(sel[~sel["video"].str.startswith("2023-05-02")]["filepath"].to_numpy(), 24, replace=False).tolist()
grid(sample_match, tag="match ")

In [ ]:
sample_drill = rng.choice(sel[sel["video"].str.startswith("2023-05-02")]["filepath"].to_numpy(), 12, replace=False).tolist()
grid(sample_drill, tag="drill ")

### The uncertain ones

Clips with only one player, or a second player with a low score, are the ones most likely to be wrong.

In [ ]:
one = sel[sel["n_persons"] < 2]
low = sel[(sel["n_persons"] == 2) & (sel["p1_score"] < sel["p1_score"].quantile(0.03))]
print(f"{len(one)} clips with fewer than two players; {len(low)} with the lowest 3% second player scores")
grid(one.sample(min(8, len(one)), random_state=0)["filepath"].tolist(), tag="one player ")

In [ ]:
grid(low.sample(min(8, len(low)), random_state=0)["filepath"].tolist(), tag="low score ")

### What to take from this

- Count the mistakes in the match sample: a grey box that is clearly a player but not coloured, or a colour on a bystander. If only a few of 24 are wrong, the selection is good enough to run pose on.
- The drill session is expected to be harder (other people practising on court).
- Next: `extract_selected_poses.py` runs RTMPose on the chosen boxes only, and notebook 32 trains ST-GCN on near only, far only and both players.